# 02. 전처리

`01_data_merge.ipynb`에서 만든 `merged_data.csv`를 분석 가능한 형태로 정리한다.
- 결측치, 중복, 병합 누락 여부를 다시 확인한다.
- 범죄율과 인구 대비 비율 변수를 만든다.
- 리뷰용 최종 데이터와 회귀분석용 표준화 데이터를 따로 저장한다.


In [1]:
# 데이터 분석에 필요한 도구(pandas, numpy 등)를 불러옴
# 입력 파일(merged_data.csv)과 결과로 저장할 파일들의 경로를 미리 정해둠
# 노트북을 어디서 실행해도 같은 파일을 찾을 수 있도록 프로젝트 폴더 경로를 고정함
from pathlib import Path
# Path : 폴더/파일 경로를 객체로 다루어, '/' 기호로 경로를 쉽게 이어붙일 수 있게 해주는 도구

import numpy as np    # 숫자 계산(평균, 무한대 확인 등)을 위한 라이브러리
import pandas as pd   # 표(DataFrame) 형태로 데이터를 다루는 라이브러리

# 노트북 실행 위치가 달라도 같은 파일을 읽도록 프로젝트 루트를 고정한다.
PROJECT_ROOT = Path('/Users/ijaejun/Documents/sophomore_high/crime_catchers')
# Path(경로문자열) : 문자열 경로를 Path 객체로 만들어줌
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
# Path 객체에서 '/' 연산자는 폴더를 이어붙이는 역할을 함 (PROJECT_ROOT 아래의 data/processed 폴더)
MERGED_PATH = DATA_DIR / 'merged_data.csv'
FINAL_PATH = DATA_DIR / 'final_data.csv'
MODEL_PATH = DATA_DIR / 'model_data_scaled.csv'

print('프로젝트 위치:', PROJECT_ROOT)
print('입력 파일:', MERGED_PATH)


프로젝트 위치: /Users/ijaejun/Documents/sophomore_high/crime_catchers
입력 파일: /Users/ijaejun/Documents/sophomore_high/crime_catchers/data/processed/merged_data.csv


## 1. 데이터 로드 및 기본 구조 확인

In [2]:
# ── 함수 설명 ──────────────────
# 01_data_merge.ipynb에서 만든 merged_data.csv를 불러와서, 분석에 필요한 컬럼이 모두 있는지 점검한다.
df = pd.read_csv(MERGED_PATH, encoding='utf-8-sig')
# read_csv(): CSV 파일을 표(DataFrame)로 읽어오는 함수
#   └ MERGED_PATH : 읽어올 파일의 경로
#   └ encoding='utf-8-sig' : 한글이 깨지지 않도록 인코딩 방식을 지정

CRIME_COLUMNS = ['살인기수', '강도', '강간', '절도', '폭행']
# 5대 범죄 각각의 건수가 들어있는 컬럼 이름들을 리스트로 모아둠

# 병합 노트북을 다시 실행하기 전 파일에는 개별 범죄 컬럼만 있을 수 있다.
# 이 경우 전처리에서 5대범죄합계를 만들어 뒤 단계가 끊기지 않게 한다.
if '5대범죄합계' not in df.columns and set(CRIME_COLUMNS).issubset(df.columns):
# if 조건문: '5대범죄합계' 컬럼이 없고, 5개 범죄 컬럼이 모두 존재할 때만 아래 코드를 실행
#   └ not in : '~안에 없으면'을 뜻함
#   └ set(...).issubset(df.columns) : CRIME_COLUMNS의 모든 항목이 df의 컬럼에 포함되어 있는지 확인
    df[CRIME_COLUMNS] = df[CRIME_COLUMNS].apply(pd.to_numeric, errors='coerce')
    # apply(pd.to_numeric): 각 컬럼의 값을 숫자로 바꿔주는 함수
    #   └ errors='coerce' : 숫자로 바꿀 수 없는 값은 에러 대신 NaN(결측치)으로 처리
    df['5대범죄합계'] = df[CRIME_COLUMNS].sum(axis=1).astype(int)
    # sum(axis=1): 한 행(row) 안에서 여러 컬럼의 값을 가로로 더함
    #   └ axis=1 : 행 방향(가로)으로 합산 (axis=0이면 컬럼 전체를 세로로 합산)
    # astype(int): 계산 결과를 정수형으로 변환

EXPECTED_COLUMNS = [
    '지역', '연도', '5대범죄합계',
    '실업률', '음주율', '물가상승률', '인구수', '평균기온', '경찰1인당주민수',
    '기초수급자수', '조이혼율', '지역소득', '외국인수', '외국인비율(%)'
]
# 이후 전처리에 꼭 필요한 컬럼 이름들을 미리 정해둔 목록

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
# set(A) - set(B): A에는 있지만 B에는 없는 항목만 골라냄 (EXPECTED인데 df에 없는 컬럼)
#   └ sorted(...) : 결과를 가나다 순서로 정렬
extra_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))
# 반대로 df에는 있지만 EXPECTED_COLUMNS에는 없는 '예상 밖' 컬럼을 찾음

print('shape:', df.shape)
# shape : (행 개수, 열 개수)를 튜플로 알려주는 속성
print('누락 컬럼:', missing_columns)
print('추가 컬럼:', extra_columns)
print(df.head())
# head(): 데이터의 맨 위 5개 행만 보여주는 함수 (기본값 n=5)

if missing_columns:
    raise ValueError(f'필수 컬럼이 없습니다: {missing_columns}')
    # raise ValueError(...): 필수 컬럼이 없으면 오류를 강제로 발생시켜 실행을 멈추고 알려줌


shape: (66, 14)
누락 컬럼: []
추가 컬럼: []
   지역    연도  5대범죄합계  실업률   음주율  물가상승률      인구수  평균기온  경찰1인당주민수  기초수급자수  조이혼율  \
0  광주  2014   16364  2.8  60.6    1.6  1475884  14.3     476.7   59598   2.1   
1  광주  2015   14095  2.9  61.9    0.3  1472199  14.6     458.6   71683   1.9   
2  광주  2016   11073  3.1  58.6    0.9  1469214  15.0     454.6   69420   1.9   
3  광주  2017    9916  2.9  61.6    2.1  1463770  14.6     447.5   65712   1.8   
4  광주  2018   10070  3.8  60.3    1.2  1459336  14.6     439.6   72757   2.0   

    지역소득   외국인수  외국인비율(%)  
0  23448  17064    1.1562  
1  24731  18455    1.2536  
2  26248  19920    1.3558  
3  27449  21279    1.4537  
4  28594  22815    1.5634  


## 2. 품질 점검

`inner merge` 결과만 보면 누락 행이 숨어 있을 수 있으므로, 전처리 시작 전에 현재 파일 기준의 행 수와 결측/중복을 명확히 확인한다.


In [3]:
# ── 함수 설명 ──────────────────
# 6개 광역시 x 2014~2024년 = 66행이어야 한다.
expected_rows = 6 * 11
# 6개 도시 × 11개 연도(2014~2024) = 66, 데이터 행 개수가 이 값과 일치해야 정상

key_duplicates = df.duplicated(subset=['지역', '연도']).sum()
# duplicated(): 행이 중복되었는지 True/False로 알려주는 함수
#   └ subset=['지역','연도'] : '지역'과 '연도'가 모두 같은 행을 중복으로 판단
# sum(): True를 1, False를 0으로 보고 모두 더해서 중복된 행의 개수를 셈

missing_by_column = df.isna().sum()
# isna(): 각 칸이 결측치(빈 값)인지 True/False로 표시하는 함수
# sum(): 컬럼별로 True(결측치)의 개수를 합산

print('예상 행 수:', expected_rows)
print('실제 행 수:', len(df))
# len(df) : DataFrame의 행 개수를 반환하는 함수
print('지역-연도 중복 수:', key_duplicates)
print()
print('결측치 개수:')
print(missing_by_column)

if len(df) != expected_rows:
    raise ValueError(f'행 수가 예상과 다릅니다. expected={expected_rows}, actual={len(df)}')
    # 행 개수가 66이 아니면 데이터가 빠지거나 중복됐다는 뜻이므로 에러로 알려줌
if key_duplicates > 0:
    raise ValueError('지역-연도 기준 중복 행이 있습니다.')
if missing_by_column.sum() > 0:
    raise ValueError('결측치가 있습니다. 먼저 원본/병합 과정을 확인하세요.')


예상 행 수: 66
실제 행 수: 66
지역-연도 중복 수: 0

결측치 개수:
지역          0
연도          0
5대범죄합계      0
실업률         0
음주율         0
물가상승률       0
인구수         0
평균기온        0
경찰1인당주민수    0
기초수급자수      0
조이혼율        0
지역소득        0
외국인수        0
외국인비율(%)    0
dtype: int64


In [4]:
# ── 함수 설명 ──────────────────
# 문자열로 들어온 '-' 또는 빈칸은 isna()에서 안 잡힐 수 있어서 별도로 확인한다.
MISSING_MARKERS = {'', '-', 'NA', 'N/A', 'null', 'NULL', 'None', 'nan'}
# 결측치를 의미할 수 있는 문자열 표시들을 모아놓은 집합(set)

marker_counts = {}
# 컬럼별로 숨은 결측 표시가 몇 개인지 저장할 빈 딕셔너리

for col in df.columns:
# for문: df의 모든 컬럼 이름을 하나씩 col에 담아 반복
    values = df[col].astype(str).str.strip()
    # astype(str): 값을 모두 문자열(글자)로 바꿈
    # str.strip(): 문자열 앞뒤의 빈 공백을 제거
    count = values.isin(MISSING_MARKERS).sum()
    # isin(MISSING_MARKERS): 값이 MISSING_MARKERS 안에 있으면 True
    # sum(): True의 개수(=숨은 결측 표시 개수)를 셈
    if count > 0:
        marker_counts[col] = int(count)
        # 결측 표시가 1개 이상이면 marker_counts 딕셔너리에 {컬럼명: 개수} 형태로 저장

numeric_columns = [col for col in EXPECTED_COLUMNS if col not in ['지역']]
# 리스트 컴프리헨션: EXPECTED_COLUMNS 중 '지역'을 제외한 나머지(숫자 컬럼)만 골라 새 리스트로 만듦
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')
# apply(pd.to_numeric, errors='coerce'): 각 컬럼을 숫자로 변환, 실패하면 NaN으로 처리
inf_count = np.isinf(df[numeric_columns].to_numpy()).sum()
# to_numpy(): DataFrame을 숫자 배열(array)로 변환
# np.isinf(): 값이 무한대(inf)인지 True/False로 확인
# sum(): True(무한대인 칸)의 개수를 셈

print('숨은 결측 표시:', marker_counts)
print('숫자 변환 후 결측치 합계:', int(df[numeric_columns].isna().sum().sum()))
# isna().sum().sum() : 먼저 컬럼별 결측치 개수를 구하고(첫 sum), 그 합계를 다시 더함(둘째 sum)
print('inf 개수:', int(inf_count))

if marker_counts:
    raise ValueError(f'문자열 결측 표시가 있습니다: {marker_counts}')
if df[numeric_columns].isna().sum().sum() > 0:
    raise ValueError('숫자 변환 과정에서 결측치가 생겼습니다.')
if inf_count > 0:
    raise ValueError('무한대 값이 있습니다. 인구수 0 여부 등을 확인하세요.')
    # 인구수가 0이면 '값/인구수' 계산에서 나누기 0이 발생해 inf가 생길 수 있음


숨은 결측 표시: {}
숫자 변환 후 결측치 합계: 0
inf 개수: 0


## 3. 분석용 변수 생성

README 기준으로 종속변수는 인구 10만 명당 범죄율로 통일한다. 외국인수와 기초수급자수는 도시 인구 규모의 영향을 줄이기 위해 비율 변수로 바꾼다.


In [5]:
# ── 함수 설명 ──────────────────
# 종속변수(범죄율)와 독립변수(기초수급비율)를 새로 계산해서 추가하는 코드
final_df = df.copy()
# copy(): 원본 df를 건드리지 않도록 복사본을 만듦 (복사본에서만 수정)

# 종속변수(Y): 도시 규모 차이를 줄이기 위해 인구 10만 명당 범죄 발생 건수로 변환한다.
final_df['범죄율'] = (final_df['5대범죄합계'] / final_df['인구수'] * 100000).round(2)
# 새 컬럼 만들기: final_df['범죄율'] = ... 처럼 없는 컬럼명에 값을 대입하면 새 컬럼이 생성됨
#   └ 5대범죄합계 ÷ 인구수 × 100000 : 인구 10만 명당 범죄 건수로 환산 (도시마다 인구가 달라도 비교 가능)
# round(2): 소수점 둘째 자리까지만 남기고 나머지는 반올림

# 독립변수(X): 기초수급자수는 절대값 대신 인구 대비 비율을 사용한다.
final_df['기초수급비율(%)'] = (final_df['기초수급자수'] / final_df['인구수'] * 100).round(4)
#   └ 기초수급자수 ÷ 인구수 × 100 : 인구 대비 기초생활수급자 비율(%)
# round(4): 소수점 넷째 자리까지 반올림

# 인구수는 외국인비율(%), 기초수급비율(%) 계산에만 사용하고
# 절대값 자체는 분석에 불필요하므로 최종 컬럼에서 제외한다.
ANALYSIS_COLUMNS = [
    '지역', '연도', '범죄율',
    '실업률', '음주율', '물가상승률', '평균기온', '경찰1인당주민수',
    '기초수급비율(%)', '조이혼율', '지역소득', '외국인비율(%)'
]
# 최종 분석에 사용할 컬럼 이름만 모아놓은 목록 (종속변수 1개 + 독립변수 9개 + 지역/연도)

final_df = final_df[ANALYSIS_COLUMNS].sort_values(['지역', '연도']).reset_index(drop=True)
# final_df[ANALYSIS_COLUMNS] : 목록에 있는 컬럼만 선택
# sort_values(['지역','연도']): '지역' 기준으로 먼저 정렬하고, 같은 지역 안에서는 '연도' 기준으로 정렬
# reset_index(drop=True): 정렬 후 뒤섞인 행 번호(인덱스)를 0부터 다시 매김
#   └ drop=True : 기존 인덱스를 새 컬럼으로 남기지 않고 버림

print('분석용 데이터 shape:', final_df.shape)
print(final_df.head(10))
# head(10): 위에서부터 10개 행만 보여줌


분석용 데이터 shape: (66, 12)
   지역    연도      범죄율  실업률   음주율  물가상승률  평균기온  경찰1인당주민수  기초수급비율(%)  조이혼율  \
0  광주  2014  1108.76  2.8  60.6    1.6  14.3     476.7     4.0381   2.1   
1  광주  2015   957.41  2.9  61.9    0.3  14.6     458.6     4.8691   1.9   
2  광주  2016   753.67  3.1  58.6    0.9  15.0     454.6     4.7250   1.9   
3  광주  2017   677.43  2.9  61.6    2.1  14.6     447.5     4.4892   1.8   
4  광주  2018   690.04  3.8  60.3    1.2  14.6     439.6     4.9856   2.0   
5  광주  2019   710.76  3.7  61.1    0.2  14.7     423.8     5.2314   2.0   
6  광주  2020   646.87  3.9  52.9    0.4  14.5     419.0     5.8454   1.8   
7  광주  2021   599.47  3.6  54.5    2.6  15.1     404.0     6.3504   1.8   
8  광주  2022   614.72  2.9  58.6    5.1  14.8     401.0     6.5782   1.6   
9  광주  2023   684.24  2.5  59.5    3.7  15.3     398.9     6.7975   1.7   

    지역소득  외국인비율(%)  
0  23448    1.1562  
1  24731    1.2536  
2  26248    1.3558  
3  27449    1.4537  
4  28594    1.5634  
5  29807    1.6358  
6  

## 4. 최종 검증 및 저장

In [6]:
# ── 함수 설명 ──────────────────
# 최종 데이터에 문제(결측치, inf)가 없는지 마지막으로 확인하고 파일로 저장하는 코드
final_missing = final_df.isna().sum()
# isna().sum(): 컬럼별 결측치 개수를 구함
final_inf = np.isinf(final_df.select_dtypes(include='number').to_numpy()).sum()
# select_dtypes(include='number'): 숫자형 컬럼만 골라냄 ('지역'처럼 글자 컬럼은 제외)
# to_numpy(): 숫자 배열로 변환 후 np.isinf()로 무한대 여부 확인, sum()으로 개수 합산

print('최종 결측치:')
print(final_missing)
print('최종 inf 개수:', int(final_inf))

if final_missing.sum() > 0:
    raise ValueError('최종 데이터에 결측치가 남아 있습니다.')
if final_inf > 0:
    raise ValueError('최종 데이터에 inf 값이 남아 있습니다.')

DATA_DIR.mkdir(parents=True, exist_ok=True)
# mkdir(): 폴더를 새로 만드는 함수
#   └ parents=True : 중간에 없는 상위 폴더까지 한꺼번에 생성
#   └ exist_ok=True : 폴더가 이미 있어도 에러를 내지 않음
final_df.to_csv(FINAL_PATH, index=False, encoding='utf-8-sig')
# to_csv(): DataFrame을 CSV 파일로 저장하는 함수
#   └ index=False : 행 번호(인덱스)는 파일에 저장하지 않음
#   └ encoding='utf-8-sig' : 한글이 깨지지 않도록 저장
print('저장 완료:', FINAL_PATH)


최종 결측치:
지역           0
연도           0
범죄율          0
실업률          0
음주율          0
물가상승률        0
평균기온         0
경찰1인당주민수     0
기초수급비율(%)    0
조이혼율         0
지역소득         0
외국인비율(%)     0
dtype: int64
최종 inf 개수: 0
저장 완료: /Users/ijaejun/Documents/sophomore_high/crime_catchers/data/processed/final_data.csv


## 5. 회귀분석용 데이터 만들기

회귀분석에서는 변수 단위가 다르면 계수 비교가 어려우므로 독립변수만 표준화한다. `지역`은 도시 고유 차이를 반영하기 위해 더미변수로 바꾼다.


In [7]:
# ── 함수 설명 ──────────────────
# 회귀분석에 바로 쓸 수 있도록 '지역'을 더미변수로 바꾸고, 독립변수를 표준화(z-score)하는 코드
FEATURE_COLUMNS = [
    '실업률', '음주율', '물가상승률', '평균기온', '경찰1인당주민수',
    '기초수급비율(%)', '조이혼율', '지역소득', '외국인비율(%)'
]
# 표준화(z-score)를 적용할 독립변수 9개의 컬럼 이름 목록

model_df = pd.get_dummies(
    final_df,
    columns=['지역'],
    prefix='지역',
    drop_first=True,
    dtype=int
)
# get_dummies(): 글자(범주형) 컬럼을 0/1 숫자 컬럼들로 바꿔주는 함수 (원-핫 인코딩)
#   └ columns=['지역'] : '지역' 컬럼만 더미변수로 변환
#   └ prefix='지역' : 새로 생기는 컬럼 이름 앞에 '지역_' 을 붙임 (예: 지역_부산)
#   └ drop_first=True : 첫 번째 지역(기준 도시)은 컬럼을 만들지 않음 (다중공선성 방지)
#   └ dtype=int : 결과값을 0/1 정수로 만듦

# sklearn이 없어도 실행되도록 pandas로 z-score 표준화를 직접 계산한다.
feature_mean = model_df[FEATURE_COLUMNS].mean()
# mean(): 각 컬럼(변수)의 평균값을 계산
feature_std = model_df[FEATURE_COLUMNS].std(ddof=0).replace(0, 1)
# std(ddof=0): 각 컬럼의 표준편차를 계산
#   └ ddof=0 : 모집단 기준 표준편차로 계산 (n으로 나눔)
# replace(0, 1): 표준편차가 0인 경우 0으로 나누는 오류를 막기 위해 1로 바꿔줌
model_df[FEATURE_COLUMNS] = ((model_df[FEATURE_COLUMNS] - feature_mean) / feature_std).round(6)
# (값 - 평균) / 표준편차 : z-score 표준화 공식 → 평균 0, 표준편차 1로 변환
# round(6): 소수점 여섯째 자리까지 반올림

model_df.to_csv(MODEL_PATH, index=False, encoding='utf-8-sig')
# to_csv(): 표준화된 데이터를 model_data_scaled.csv 파일로 저장

print('모델용 데이터 shape:', model_df.shape)
print('저장 완료:', MODEL_PATH)
print(model_df.head())


모델용 데이터 shape: (66, 16)
저장 완료: /Users/ijaejun/Documents/sophomore_high/crime_catchers/data/processed/model_data_scaled.csv
     연도      범죄율       실업률       음주율     물가상승률      평균기온  경찰1인당주민수  기초수급비율(%)  \
0  2014  1108.76 -1.369069  0.216556 -0.185520 -0.056153  0.780781  -0.369733   
1  2015   957.41 -1.206261  0.624920 -1.105612  0.215025  0.403345   0.199085   
2  2016   753.67 -0.880644 -0.411695 -0.680954  0.576595  0.319933   0.100449   
3  2017   677.43 -1.206261  0.530682  0.168362  0.215025  0.171878  -0.060956   
4  2018   690.04  0.259013  0.122319 -0.468625  0.215025  0.007141   0.278830   

       조이혼율      지역소득  외국인비율(%)  지역_대구  지역_대전  지역_부산  지역_울산  지역_인천  
0  0.492649 -0.831983 -0.913365      0      0      0      0      0  
1 -0.310187 -0.750253 -0.704951      0      0      0      0      0  
2 -0.310187 -0.653616 -0.486266      0      0      0      0      0  
3 -0.711605 -0.577109 -0.276783      0      0      0      0      0  
4  0.091231 -0.504169 -0.042050      0      0

## 6. 코드 리뷰용 체크포인트

- `final_data.csv`: 사람이 해석하기 쉬운 전처리 완료 데이터
- `model_data_scaled.csv`: 회귀분석에 바로 넣기 좋은 표준화 + 지역 더미 데이터
- 2020~2021년은 README 방침대로 제거하지 않고 포함한다.
